# Lab 15b: Foundry-Hosted Agents with Tracing

This notebook demonstrates how to trace Foundry-hosted agents using the Azure AI Projects SDK.

## Prerequisites

Run `main.ipynb` first to deploy the required resources.

## What You'll Learn

| Concept | Description |
|---------|-------------|
| **SDK Instrumentation** | Enable tracing with `AIProjectInstrumentor` |
| **OpenTelemetry Setup** | Configure Azure Monitor exporter |
| **Agent Creation** | Create agents with `PromptAgentDefinition` |
| **Automatic Tracing** | Traces captured for all SDK operations |
| **Trace Verification** | Query App Insights to verify trace arrival |

## References

- [Agent Tracing Overview](https://learn.microsoft.com/azure/ai-foundry/observability/concepts/trace-agent-concept?view=foundry)
- [Tracing Integrations](https://learn.microsoft.com/azure/ai-foundry/observability/how-to/trace-agent-framework?view=foundry)

In [1]:
# Enable automatic masking of Azure resource names in print output
import sys
sys.path.insert(0, "../")
from secure_print import install
install()

secure_print: Azure resource masking enabled


In [2]:
%pip install -q azure-ai-projects==2.0.0b2 azure-identity openai rich
%pip install -q azure-monitor-opentelemetry azure-monitor-query


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import subprocess
import base64
import uuid
from pathlib import Path

# Load configuration from infrastructure deployment
config_file = Path("spoke-config.json")
if not config_file.exists():
    raise FileNotFoundError("Run main.ipynb first to deploy resources")

config = json.loads(config_file.read_text())

PROJECT_ENDPOINT = config["PROJECT_ENDPOINT"]
APP_INSIGHTS_CONN_STRING = config["APP_INSIGHTS_CONN_STRING"]
APP_INSIGHTS_NAME = config["APP_INSIGHTS_NAME"]
RESOURCE_GROUP = config["RESOURCE_GROUP"]
PROJECT_NAME = config["PROJECT_NAME"]
ACCOUNT_NAME = config["ACCOUNT_NAME"]

# Model needs connection prefix for APIM routing
MODEL_NAME = config["GATEWAY_MODEL"]
GATEWAY_MODEL = f"landing-zone-apim/{MODEL_NAME}"

# Get subscription ID for building portal links
sub_result = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True)
SUBSCRIPTION_ID = sub_result.stdout.strip()

# Agent configuration
AGENT_NAME = "NASASpaceFactsAgent"

print("Configuration loaded")
print(f"  Project: {PROJECT_NAME}")
print(f"  Model: {GATEWAY_MODEL}")

Configuration loaded
  Project: tracing-project
  Model: landing-zone-apim/gpt-4.1-mini


---
## Step 1: Configure OpenTelemetry with Azure Monitor

The `configure_azure_monitor()` function sets up all OpenTelemetry components automatically:
- TracerProvider with Azure Monitor exporter
- MetricProvider (optional)
- LoggerProvider (optional)

This must be called **before** instrumenting the SDK.

In [4]:
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider

# Check if already configured
current_provider = otel_trace.get_tracer_provider()
if isinstance(current_provider, TracerProvider):
    provider = current_provider
    print("Reusing existing TracerProvider")
else:
    # Configure Azure Monitor with all OpenTelemetry components
    configure_azure_monitor(connection_string=APP_INSIGHTS_CONN_STRING)
    provider = otel_trace.get_tracer_provider()
    print("OpenTelemetry configured with Azure Monitor")

print("TracerProvider ready")

OpenTelemetry configured with Azure Monitor
TracerProvider ready


---
## Step 2: Instrument Azure AI Projects SDK

The SDK must be **explicitly instrumented** to enable tracing:

```python
from azure.ai.projects.telemetry import AIProjectInstrumentor  # Note: singular, no 's'
AIProjectInstrumentor().instrument(enable_content_recording=True)
```

This must happen **before** creating any clients.

In [5]:
from azure.ai.projects.telemetry import AIProjectInstrumentor

AIProjectInstrumentor().instrument(enable_content_recording=True)

print("Azure AI Projects SDK instrumented")
print("  Content recording: enabled")

Azure AI Projects SDK instrumented
  Content recording: enabled


---
## Step 3: Create Foundry Client

Now that instrumentation is enabled, all SDK operations will be automatically traced.

In [6]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool
from azure.identity import DefaultAzureCredential

# Initialize the Foundry client (now instrumented)
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

# Get OpenAI client for Responses API
openai_client = project_client.get_openai_client()

print("Connected to Foundry project")
print(f"  Endpoint: {PROJECT_ENDPOINT[:50]}...")

Connected to Foundry project
  Endpoint: https://tra***.services.ai.azure.com...


---
## Step 4: Create a NASA Space Facts Agent

We'll create a space-themed agent using `PromptAgentDefinition`. The agent creation itself generates traces.

In [7]:
# Define NASA Space Facts agent with function calling
SPACE_AGENT_INSTRUCTIONS = """You are a NASA Space Facts expert assistant.
Your mission is to share fascinating facts about space exploration, planets, stars, 
galaxies, and NASA missions. When users ask about specific celestial bodies or missions,
use the get_space_fact function to provide detailed information.

Be enthusiastic and educational. Include fun comparisons to help users understand scale.
Use emojis to make responses engaging."""

space_fact_function = FunctionTool(
    name="get_space_fact",
    description="Get detailed information about a space topic, planet, or NASA mission",
    parameters={
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The space topic to get facts about (e.g., 'Mars', 'James Webb', 'black holes')"
            },
            "category": {
                "type": "string",
                "enum": ["planet", "mission", "phenomenon", "general"],
                "description": "Category of the space fact"
            }
        },
        "required": ["topic"]
    }
)

# Create the agent (this operation is traced)
space_agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=GATEWAY_MODEL,
        instructions=SPACE_AGENT_INSTRUCTIONS,
        tools=[space_fact_function],
    ),
)

AGENT_VERSION = space_agent.version

print(f"Created agent: {space_agent.name} v{AGENT_VERSION}")
print(f"  Model: {GATEWAY_MODEL}")
print(f"  Tools: get_space_fact")

Created agent: NASASpaceFactsAgent v2
  Model: landing-zone-apim/gpt-4.1-mini
  Tools: get_space_fact


---
## Step 5: Invoke the Agent (Automatically Traced)

When we invoke this agent, Foundry automatically captures:
- Request/response data
- Tool calls and results
- Latency and token usage
- Error information (if any)

Let's ask some space questions!

In [8]:
import time

# Invoke the agent multiple times to generate traces
queries = [
    "Tell me a fascinating fact about Mars!",
    "What is the James Webb Space Telescope discovering?",
    "How big is the largest known black hole?",
]

print("Invoking NASA Space Facts agent (traces captured automatically)...\n")

for query in queries:
    print(f"User: {query}")
    
    start = time.time()
    response = openai_client.responses.create(
        input=query,
        extra_body={
            "agent": {
                "name": AGENT_NAME,
                "version": AGENT_VERSION,
                "type": "agent_reference"
            }
        },
    )
    duration = (time.time() - start) * 1000
    
    # Extract response text
    output_text = getattr(response, 'output_text', None) or str(response.output)
    print(f"Agent: {output_text[:300]}")
    print(f"  Duration: {duration:.0f}ms\n")
    
    time.sleep(1)  # Small delay between requests

print("Agent invocations complete!")

Invoking NASA Space Facts agent (traces captured automatically)...

User: Tell me a fascinating fact about Mars!
Agent: [ResponseFunctionToolCall(arguments='{"topic":"Mars","category":"planet"}', call_id='call_sHEMSpMjES4QAROB7bZ90J3U', name='get_space_fact', type='function_call', id='fc_0bd5***adb3', status='completed', created_by={'agent': {'type': 'agent_id', 'name': 'NASASpa
  Duration: 2734ms

User: What is the James Webb Space Telescope discovering?
Agent: [ResponseFunctionToolCall(arguments='{"topic":"James Webb","category":"mission"}', call_id='call_lzoZLJxdij9qYDi7WmaUt93b', name='get_space_fact', type='function_call', id='fc_0bf2***de8a', status='completed', created_by={'agent': {'type': 'agent_id', 'name': '
  Duration: 3497ms

User: How big is the largest known black hole?
Agent: [ResponseFunctionToolCall(arguments='{"topic":"largest known black hole","category":"phenomenon"}', call_id='call_V4nPB7yAjQkeMRhok8Yytulg', name='get_space_fact', type='function_call', id='fc_075d

---
## Step 6: Flush and Verify Traces

OpenTelemetry batches traces for efficiency. We flush to ensure all traces are sent, then verify arrival in App Insights.

In [9]:
import subprocess

# Force flush all pending traces
print("Flushing traces to Application Insights...")
provider.force_flush()
print("Traces flushed\n")

# Wait for ingestion (App Insights has ~30 second delay)
print("Waiting 45 seconds for trace ingestion...")
time.sleep(45)

Flushing traces to Application Insights...
Traces flushed

Waiting 45 seconds for trace ingestion...


In [10]:
from datetime import timedelta
from azure.monitor.query import LogsQueryClient
from azure.identity import DefaultAzureCredential

print(f"Querying Application Insights: {APP_INSIGHTS_NAME}\n")

logs_client = LogsQueryClient(credential=DefaultAzureCredential())

resource_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/microsoft.insights/components/{APP_INSIGHTS_NAME}"

# Query for agent traces (LLM calls and tool executions) - last 30 minutes
deps_query = """
dependencies
| where timestamp > ago(30m)
| where name contains "chat" or name contains "execute_tool" or name contains "agent"
| project timestamp, name, duration, success
| order by timestamp desc
| take 10
"""

try:
    dep_response = logs_client.query_resource(
        resource_id=resource_id,
        query=deps_query,
        timespan=timedelta(minutes=30)
    )
    
    if dep_response.tables and len(dep_response.tables[0].rows) > 0:
        print(f"Found {len(dep_response.tables[0].rows)} AI operation(s) in last 15 minutes:\n")
        for row in dep_response.tables[0].rows:
            ts = str(row[0])[11:19] if row[0] else "N/A"  # Just time portion
            name = str(row[1])[:35] if row[1] else "N/A"
            duration = f"{row[2]:.0f}ms" if row[2] else "N/A"
            status = "OK" if row[3] else "FAIL"
            print(f"  {ts} | {name} | {duration} | {status}")
        print("\nTraces verified in Application Insights!")
    else:
        print("  No AI operations found in last 15 minutes - wait a bit longer")
except Exception as e:
    print(f"Error querying: {e}")

Querying Application Insights: tracing-insights-xx43tl

Found 10 AI operation(s) in last 15 minutes:

  18:38:31 | execute_tool functions.get_space_fa | 6ms | OK
  18:38:30 | chat landing-zone-apim/gpt-4.1-mini | 744ms | OK
  18:38:30 | responses NASASpaceFactsAgent | 1310ms | OK
  18:38:29 | execute_tool functions.get_space_fa | 11ms | OK
  18:38:28 | chat landing-zone-apim/gpt-4.1-mini | 835ms | OK
  18:38:25 | responses NASASpaceFactsAgent | 3496ms | OK
  18:38:24 | execute_tool functions.get_space_fa | 9ms | OK
  18:38:23 | chat landing-zone-apim/gpt-4.1-mini | 976ms | OK
  18:38:22 | responses NASASpaceFactsAgent | 2732ms | OK
  18:38:21 | POST /api/projects/tracing-project/ | 1066ms | OK

Traces verified in Application Insights!


---
## Step 7: View Your Traces

Now that traces have been captured, let's generate direct links to view them in both the Foundry Portal and Azure Portal.

In [ ]:
# Generate exact portal links for viewing traces

# Foundry Portal URL (encoded subscription ID format)
sub_bytes = uuid.UUID(SUBSCRIPTION_ID).bytes
encoded_sub = base64.urlsafe_b64encode(sub_bytes).decode('utf-8').rstrip('=')

FOUNDRY_TRACES_URL = (
    f"https://ai.azure.com/nextgen/r/{encoded_sub},{RESOURCE_GROUP},,{ACCOUNT_NAME},{PROJECT_NAME}"
    f"/build/agents/{AGENT_NAME}/traces?version={AGENT_VERSION}"
)

# Azure Portal - App Insights Agents Dashboard
TENANT_RESULT = subprocess.run('az account show --query tenantId -o tsv', shell=True, capture_output=True, text=True)
TENANT_ID = TENANT_RESULT.stdout.strip()

APP_INSIGHTS_AGENTS_URL = (
    f"https://ms.portal.azure.com/#@{TENANT_ID}/resource"
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.Insights/components/{APP_INSIGHTS_NAME}/agents"
)

# App Insights Logs URL
APP_INSIGHTS_LOGS_URL = (
    f"https://ms.portal.azure.com/#@{TENANT_ID}/resource"
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.Insights/components/{APP_INSIGHTS_NAME}/logs"
)

print("="*70)
print("WHERE TO VIEW YOUR TRACES")
print("="*70)

print("\n1. FOUNDRY PORTAL - Agent Traces")
print("   View traces for your specific agent with conversation details:")
print(f"   {FOUNDRY_TRACES_URL}")

print("\n2. AZURE PORTAL - Agents Dashboard")
print("   View aggregated agent metrics and performance:")
print(f"   {APP_INSIGHTS_AGENTS_URL}")


---
## Summary

| Step | What Happens |
|------|---------------|
| 1. Configure OpenTelemetry | `configure_azure_monitor(connection_string=...)` |
| 2. Instrument SDK | `AIProjectInstrumentor().instrument()` - must be before clients |
| 3. Create Client | `AIProjectClient` - now all operations are traced |
| 4. Create Agent | `NASASpaceFactsAgent` with `PromptAgentDefinition` |
| 5. Invoke Agent | Ask space questions - invocations are traced |
| 6. Verify | Query App Insights to confirm trace arrival |
| 7. View | Open Foundry Portal or App Insights to explore traces |